# Mini-projet : Intégration de l'IA MCP + Agents dans Gemini

**Objectif :** Créer une application multi-agents complète qui orchestre plusieurs serveurs MCP (Model Context Protocol) à l'aide d'un LLM (Gemini) et d'une politique flexible pilotée par outil — c'est le LLM qui détermine les prochaines étapes, et non un flux prédéfini.

**Environnement cible :** Google Colab.

---

## Thème choisi : « Assistant d'espace de travail »

L'agent orchestre trois serveurs MCP :
- **`filesystem`** (serveur MCP officiel) — lecture/écriture de fichiers dans un répertoire de travail.
- **`git`** (serveur MCP officiel) — consultation de l'historique et du statut d'un dépôt Git.
- **`custom_ops`** (serveur MCP personnalisé, écrit avec FastMCP) — outils de résumé et d'analyse de texte sur mesure.

L'agent Gemini reçoit une instruction en langage naturel (ex. *« résume le fichier notes.md et indique le dernier commit »*) et décide lui-même, tour après tour, quels outils appeler et dans quel ordre.

## 1. Configuration de Colab : installation des dépendances

In [ ]:
%pip install -qU \

  "langchain>=0.3" \

  "langgraph>=0.2" \

  "langchain-google-genai>=2.0" \

  "google-genai>=1.0" \

  "langchain-mcp-adapters==0.2.1" \

  "nest_asyncio"

**Notes**
- `langchain-google-genai` est l'intégration LangChain pour Gemini.
- `langchain-mcp-adapters` fournit un client MCP compatible avec les outils LangChain.

## 2. Configurer `GOOGLE_API_KEY` dans Colab

In [ ]:
import os

from getpass import getpass



if "GOOGLE_API_KEY" not in os.environ or not os.environ["GOOGLE_API_KEY"]:

    os.environ["GOOGLE_API_KEY"] = getpass("Entrez votre GOOGLE_API_KEY : ")



print("GOOGLE_API_KEY défini :", bool(os.environ.get("GOOGLE_API_KEY")))

## 3. Vérifier la disponibilité de Node/NPM (requis pour de nombreux serveurs MCP)

In [ ]:
!node --version

!npx --version

Si Node/NPM manquent (typiquement hors Colab, ou image minimale) :

In [ ]:
!apt-get -qq update

!apt-get -qq install -y nodejs npm

!node --version

!npx --version

## 4. Choisir des serveurs MCP tiers

**Exigence :** utiliser au moins deux serveurs MCP tiers.

Pour ce projet :
1. `@modelcontextprotocol/server-filesystem` — accès fichiers (package npm officiel).
2. `mcp-server-git` — accès Git (package Python officiel).

La liste complète des serveurs MCP officiels est disponible sur le dépôt GitHub `modelcontextprotocol/servers`.

In [ ]:
# Installation du serveur Git (package Python)

%pip install -qU mcp-server-git

### Préparer un répertoire de travail (dépôt Git de démonstration)

On crée un petit dépôt Git avec quelques fichiers pour que les outils `filesystem` et `git` aient une matière concrète sur laquelle travailler.

In [ ]:
import subprocess

from pathlib import Path



WORKDIR = "/content/workspace"

Path(WORKDIR).mkdir(parents=True, exist_ok=True)



notes_path = Path(WORKDIR) / "notes.md"

notes_path.write_text(

    "# Notes de projet\n\n"

    "- Mettre en place le client MCP multi-serveurs.\n"

    "- Connecter le serveur filesystem et le serveur git.\n"

    "- Ecrire un serveur MCP personnalise avec FastMCP.\n"

    "- Tester l'agent Gemini de bout en bout.\n",

    encoding="utf-8",

)



subprocess.run(["git", "init"], cwd=WORKDIR, check=True)

subprocess.run(["git", "config", "user.email", "demo@example.com"], cwd=WORKDIR, check=True)

subprocess.run(["git", "config", "user.name", "Demo"], cwd=WORKDIR, check=True)

subprocess.run(["git", "add", "."], cwd=WORKDIR, check=True)

subprocess.run(["git", "commit", "-m", "Initial commit: ajout de notes.md"], cwd=WORKDIR, check=True)



print("Répertoire de travail prêt :", WORKDIR)

## 5. Lancer les serveurs MCP dans Colab (transport stdio)

Dans MCP, l'agent communique avec les serveurs via **stdio** (sous-processus). Dans Colab, cela signifie que l'agent lance des sous-processus exécutant :

```
npx -y <server-package> ...
python -m <server-module> ...
```

Ces sous-processus sont démarrés automatiquement par `MultiServerMCPClient` à l'étape suivante ; il n'est pas nécessaire de les lancer manuellement.

## 6. Se connecter aux serveurs MCP depuis l'environnement d'exécution de l'agent

In [ ]:
import asyncio

import nest_asyncio

nest_asyncio.apply()



from langchain_mcp_adapters.client import MultiServerMCPClient



mcp_connections = {

    "filesystem": {

        "transport": "stdio",

        "command": "npx",

        "args": ["-y", "@modelcontextprotocol/server-filesystem", WORKDIR],

    },

    "git": {

        "transport": "stdio",

        "command": "python",

        "args": ["-m", "mcp_server_git", "--repository", WORKDIR],

    },

}



client = MultiServerMCPClient(mcp_connections, tool_name_prefix=True)

## 7. Implémenter un serveur MCP personnalisé en Python (FastMCP)

Ce serveur `custom_ops` expose des outils de résumé et d'analyse de texte, complémentaires aux serveurs `filesystem` et `git`.

In [ ]:
%pip install -qU "fastmcp>=2.0.0"

In [ ]:
from pathlib import Path

import textwrap



server_path = Path("/content/custom_mcp_server.py")

server_path.write_text(textwrap.dedent('''

    from fastmcp import FastMCP

    from typing import Dict, List

    import re



    mcp = FastMCP(name="custom_ops")



    @mcp.tool

    def ping() -> str:

        """Health check tool."""

        return "pong"



    @mcp.tool

    def summarize_lines(lines: List[str]) -> Dict[str, int]:

        """Retourne des statistiques simples sur une liste de lignes de texte."""

        total = len(lines)

        nonempty = sum(1 for l in lines if l.strip())

        words = sum(len(l.split()) for l in lines)

        return {"total_lines": total, "nonempty_lines": nonempty, "total_words": words}



    @mcp.tool

    def extract_keywords(text: str, top_k: int = 5) -> List[str]:

        """Extrait les top_k mots les plus frequents (hors mots courants) d'un texte."""

        stopwords = {"le", "la", "les", "de", "des", "un", "une", "et", "a", "au", "aux",

                     "the", "a", "an", "of", "and", "to", "in", "on", "for", "with"}

        words = re.findall(r"[a-zA-Zà-ÿ]+", text.lower())

        freq: Dict[str, int] = {}

        for w in words:

            if w in stopwords or len(w) < 3:

                continue

            freq[w] = freq.get(w, 0) + 1

        ranked = sorted(freq.items(), key=lambda kv: kv[1], reverse=True)

        return [w for w, _ in ranked[:top_k]]



    if __name__ == "__main__":

        mcp.run(transport="stdio")

'''), encoding="utf-8")



print("Fichier écrit :", server_path)

## 8. Ajouter le serveur personnalisé au client MCP

In [ ]:
mcp_connections["custom_ops"] = {

    "transport": "stdio",

    "command": "python",

    "args": [str(server_path)],

}



client = MultiServerMCPClient(mcp_connections, tool_name_prefix=True)

tools = asyncio.get_event_loop().run_until_complete(client.get_tools())



print("Nombre d'outils disponibles :", len(tools))

for t in tools:

    print(" -", t.name)

## 9. Créer un agent Gemini capable d'utiliser ces outils

On utilise `create_react_agent` de LangGraph : le LLM (Gemini) reçoit la liste complète des outils MCP (filesystem + git + custom_ops) et décide lui-même, à chaque tour, quel outil appeler — il n'y a pas de flux d'exécution codé en dur.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

from langgraph.prebuilt import create_react_agent



llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)



SYSTEM_PROMPT = (

    "Tu es un assistant d'espace de travail. Tu as accès à des outils de trois familles :\n"

    "- filesystem_* : lire/écrire/lister des fichiers dans le répertoire de travail.\n"

    "- git_* : consulter le statut, l'historique et les diffs du dépôt Git.\n"

    "- custom_ops_* : résumer du texte et en extraire les mots-clés.\n"

    "Choisis toi-même les outils pertinents et l'ordre dans lequel les appeler pour répondre "

    "à la demande de l'utilisateur. Réponds toujours en français, de façon concise."

)



agent = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)

print("Agent créé avec", len(tools), "outils.")

## 10. Démonstration

On envoie une instruction en langage naturel qui nécessite de combiner plusieurs serveurs : lecture de fichier (filesystem), résumé (custom_ops) et historique (git).

In [ ]:
async def run_agent(question: str):

    result = await agent.ainvoke({"messages": [{"role": "user", "content": question}]})

    final_message = result["messages"][-1]

    print(final_message.content)

    return result



question = (

    "Lis le fichier notes.md, résume son contenu et donne ses mots-clés principaux, "

    "puis indique le message du dernier commit Git."

)



_ = asyncio.get_event_loop().run_until_complete(run_agent(question))

In [ ]:
# Deuxième exemple : demander de créer un fichier puis de vérifier le statut git

question2 = (

    "Crée un fichier resume.md dans le répertoire de travail contenant un résumé en une phrase "

    "de notes.md, puis dis-moi si le dépôt git a des changements non validés (git status)."

)

_ = asyncio.get_event_loop().run_until_complete(run_agent(question2))

## Conclusion

Ce notebook met en place :
- deux serveurs MCP tiers officiels (**filesystem**, **git**) connectés via `MultiServerMCPClient` en transport stdio ;
- un serveur MCP **personnalisé** (`custom_ops`) écrit avec FastMCP, exposant des outils de résumé et d'extraction de mots-clés ;
- un agent **Gemini** (via LangGraph `create_react_agent`) qui décide dynamiquement, à chaque tour, quels outils appeler parmi l'ensemble des serveurs — aucune séquence n'est codée en dur.

**Pistes d'extension :** ajouter un serveur MCP de recherche web, journaliser les appels d'outils pour audit, ou exposer l'agent via une petite interface (Gradio/Streamlit).